# Object Detection with VideoDB

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/preview/object_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Detect objects in video frames using VideoDB Understanding and sandbox-backed RT-DETR inference.

**Pipeline:**
1. **Create Sandbox** — start a `small` sandbox for object detection compute
2. **Understand** — extract frames and run object detection
3. **Browse** — print detected objects, confidence scores, and timestamps

> Object indexing is not required for this preview notebook. We only create and inspect an `objects` understanding.


## 🛠️ Setup

Install the SDK and connect to VideoDB.


In [ ]:
!pip install -q "git+https://github.com/Video-DB/videodb-python.git@indexing-v2"
!pip install -q python-dotenv

In [ ]:
import os
from getpass import getpass

from videodb import connect, SandboxTier

if not os.environ.get("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect()
coll = conn.get_collection()
print("✅ Connected to VideoDB")

## 🎥 Upload or Load a Video

Upload a sample video, or replace this with `coll.get_video("m-...")` to use an existing video.


In [ ]:
video = coll.upload("https://www.youtube.com/watch?v=vVlEVRKv4is")
video.play()

In [ ]:
print("Video ID:", video.id)

# Or load an existing video:
# video = coll.get_video("m-z-YOUR-VIDEO-ID")

## 🏗️ 1. Create a Small Sandbox

Object detection uses RT-DETR and belongs to the `small` sandbox tier.

If you pass `sandbox_id`, VideoDB routes the job to that sandbox. If you omit it, VideoDB can auto-pick a compatible active sandbox for the user.


In [ ]:
# Create a small sandbox for object detection compute.
sandbox = conn.create_sandbox(tier=SandboxTier.small)
print(f"Sandbox: {sandbox.id}, Status: {sandbox.status}, Tier: {sandbox.tier}")

In [ ]:
# Wait until the sandbox is active before running object detection.
sandbox.wait_for_ready(timeout=300, interval=5)
print(f"Sandbox ready: {sandbox.id}, Status: {sandbox.status}")

In [ ]:
# Optional: list your sandboxes.
all_sandboxes = conn.list_sandboxes()
for sb in all_sandboxes:
    print(f"{sb.id} | {sb.name} | {sb.tier} | {sb.status}")

## 🔎 2. Object Understanding

`video.understand()` extracts frames from the video and runs object detection on each frame.

It returns an `understanding_id` immediately — use `get_understanding()` to poll until detection is complete.


In [ ]:
# Create Object Understanding Job.
understanding_id = video.understand(
    extract=["objects"],
    config={
        "objects": {
            "model_name": "object-detection",
            "threshold": 0.5,
            "request_chunk_size": 16,
            "max_concurrent_chunks": 1,
            "sandbox_id": sandbox.id,
        }
    },
    segmentation={"type": "time", "window": "1s"},
    sampling={"frame_count": 1},
    store=True,
)

print("Understanding ID:", understanding_id)

In [ ]:
# Fetch Understanding Job (Polling).
understanding = video.get_understanding(understanding_id)
print(understanding)

## 👀 3. View Detected Objects

The result contains per-frame object detections with labels, confidence scores, bounding boxes, and timestamps.


In [ ]:
# The SDK may expose results as objects or dictionaries depending on version.
objects_result = getattr(understanding, "results", None)

if objects_result is None and isinstance(understanding, dict):
    objects_result = (
        understanding.get("data", {})
        .get("results", {})
        .get("objects", {})
        .get("data", [])
    )

print("Object understanding result:")
print(objects_result)

In [ ]:
def get_field(item, name, default=None):
    if isinstance(item, dict):
        return item.get(name, default)
    return getattr(item, name, default)

segments = objects_result or []
total_objects = sum(len(get_field(seg, "objects", []) or []) for seg in segments)
segments_with_objects = sum(1 for seg in segments if get_field(seg, "objects", []))

print(f"Total objects detected: {total_objects}")
print(f"Segments with objects: {segments_with_objects}/{len(segments)}")

for seg in segments:
    objects = get_field(seg, "objects", []) or []
    if objects:
        ts = get_field(seg, "timestamp_ms", 0) or 0
        print(f"
Sample — {ts / 1000:.1f}s: {len(objects)} object(s)")
        for obj in objects[:10]:
            label = get_field(obj, "label")
            score = get_field(obj, "score")
            bbox = get_field(obj, "bbox")
            print(f"  {label}: score={score}, bbox={bbox}")
        break

## ⚙️ Advanced Configuration

Tune object detection with `config.objects`.

| Parameter | Default | Description |
|---|---|---|
| `model_name` | `object-detection` | Product alias for RT-DETR object detection |
| `threshold` | `0.5` | Minimum detection confidence |
| `request_chunk_size` | `256` | Frames per inference-core chunk |
| `max_concurrent_chunks` | `4` | Concurrent chunks sent by inference-core |
| `sandbox_id` | optional | Use a specific active compatible sandbox |

**Segmentation / Sampling**:

| Parameter | Default | Description |
|---|---|---|
| `segmentation.window` | `"1s"` | Time window per segment |
| `sampling.frame_count` | `1` | Frames extracted per segment |


In [ ]:
# Example: run with a higher confidence threshold.
understanding_id_v2 = video.understand(
    extract=["objects"],
    config={
        "objects": {
            "model_name": "object-detection",
            "threshold": 0.7,
            "sandbox_id": sandbox.id,
        }
    },
    segmentation={"type": "time", "window": "2s"},
    sampling={"frame_count": 1},
    store=True,
)

print("Understanding ID:", understanding_id_v2)

In [ ]:
understanding_v2 = video.get_understanding(understanding_id_v2)
print(understanding_v2)

## Utilities


In [ ]:
# List all object understandings for this video.
understandings = video.list_understanding(extract="objects")
print("Object understandings:", understandings)

## 🛑 Stop Sandbox

Stop the sandbox when finished. Billing is based on sandbox runtime.


In [ ]:
sandbox.stop()
print(f"Sandbox {sandbox.id} status: {sandbox.status}")

In [ ]:
# Optionally wait for full teardown.
sandbox.wait_for_stop(timeout=120)
print(f"Sandbox {sandbox.id} final status: {sandbox.status}")